# Graph Vocabulary and Models

```{contents}
:local:
:depth: 2
```

## Vertices and Edges

```{index} graph; vertex; edge; isolated vertex
```

A graph consists of a set of **vertices** and a set of **edges** connecting them. A vertex models an item; an edge models a relationship. A facilities team might model rooms as vertices and direct passageways as edges. A dependency model might instead use tasks as vertices and prerequisite relationships as edges.

Keep the vertex set separate from the edge set. A room with no modeled passageways is still a vertex. Such an **isolated vertex** has degree zero. If you create vertices only while reading edges, you can silently lose isolated rooms, devices, or customers.

The following office-route model has six vertices and four undirected edges. Labels are symbolic identifiers, not measured physical locations. The diagram expresses the relationships, while the arrays store them.

```text
Lobby -- Hall -- Lab       Remote
          |
        Office -- Storage
```

The drawing is not a floor plan: line lengths do not represent distance, and a crossing in a drawing would not create a new vertex unless the model explicitly added one.

In [ ]:
using System;
string[] rooms = { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" };
(string A, string B)[] edges =
{
    ("Lobby", "Hall"), ("Hall", "Lab"),
    ("Hall", "Office"), ("Office", "Storage")
};
Console.WriteLine($"vertices={rooms.Length}, edges={edges.Length}");
foreach (var (a, b) in edges) Console.WriteLine($"{a} <-> {b}");
Console.WriteLine("Remote is present even though no edge touches it.");

## Directed and Undirected Relationships

```{index} directed graph; undirected graph; arc; indegree; outdegree
```

An **undirected edge** represents a two-way relationship. One logical edge between Hall and Lab connects them in both directions. A **directed edge**, also called an arc, has a source and destination; Hall → Lab does not imply Lab → Hall.

Direction depends on the question. A prerequisite arrow from Programming I to Programming II means the first must precede the second. An import arrow might instead mean that the source file depends on the destination file. State the convention before interpreting incoming and outgoing edges.

For directed graphs, **outdegree** counts outgoing edges and **indegree** counts incoming edges. A workflow approval step can have several incoming prerequisites and one outgoing next step.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] steps = { "Request", "Budget", "Approve", "Archive" };
(string From, string To)[] dependencies =
{
    ("Request", "Approve"), ("Budget", "Approve"), ("Approve", "Archive")
};
foreach (string step in steps)
{
    int incoming = dependencies.Count(e => e.To == step);
    int outgoing = dependencies.Count(e => e.From == step);
    Console.WriteLine($"{step}: in={incoming}, out={outgoing}");
}
Console.WriteLine($"reverse approval edge={dependencies.Contains(("Approve", "Request"))}");

Approve has indegree two and outdegree one. Every directed edge contributes once to the total indegree and once to the total outdegree, so each total equals E, the number of directed edges. In an undirected graph, each ordinary edge contributes one to each endpoint's degree, giving degree sum 2E. These equalities are useful representation checks; they do not prove every edge is the intended one.

## Weights and Model Assumptions

```{index} weighted graph; unweighted graph; edge weight
```

A **weighted graph** assigns a value to each edge, such as transfer time, cost, capacity, or distance. Define units and the meaning of combination. Walking times can be added along a route. Capacities generally do not add in the same way, so an algorithm that sums weights may answer the wrong business question for that model.

The route examples use nonnegative integer minutes. Zero can represent a rounded transfer time between adjacent work areas. Absence is a different state: no direct route exists. A directed weighted model could give different costs in opposite directions; an undirected one stores one shared cost.

In [ ]:
using System;
(string A, string B, int Minutes)[] routes =
{
    ("Lobby", "Hall", 4), ("Hall", "Office", 3), ("Office", "Storage", 0)
};
foreach (var (a, b, minutes) in routes) Console.WriteLine($"{a} <-> {b}: {minutes} min");
Console.WriteLine($"specified route total={routes[0].Minutes + routes[1].Minutes + routes[2].Minutes} min");

This is the cost of one specified route, not a claim that the route is fastest. Negative weights may make sense in some other domains, but this chapter's walking-time contract rejects them. Later algorithms must state their own weight assumptions; a storage representation alone does not make an optimization method valid.

## Neighbors, Degree, and Connectivity

```{index} neighbor; degree; connected graph; connected component
```

Two vertices are **adjacent** when an edge directly connects them. A vertex's neighbors are its adjacent vertices. In the simple undirected model, degree equals the number of distinct neighbors.

```text
Lobby:   Hall
Hall:    Lobby, Lab, Office
Lab:     Hall
Office:  Hall, Storage
Storage: Office
Remote:  (none)
```

Hall has degree three; Remote has degree zero. The sum of these degrees is eight, consistent with four undirected edges. A **connected graph** has a route between every pair of vertices. A **connected component** is a maximal connected group of vertices. Here five rooms form one component and Remote forms another. An isolated vertex is a component of one vertex.

Direct adjacency differs from connectivity. Lobby and Storage have no direct edge, but the model includes the route Lobby, Hall, Office, Storage. A `HasEdge` lookup cannot by itself decide whether some longer route exists. Full reachability algorithms are deferred to Chapter 25.

## Walks, Paths, and Cycles

```{index} walk; path; cycle; path length
```

A **walk** is a sequence in which each consecutive pair is joined by an edge in the required direction. A walk may repeat vertices and edges. A **path** in this chapter is a walk without repeated vertices. A **cycle** is a closed walk with no repeated edges or vertices except its start/end. In a simple undirected graph, it uses at least three edges.

`Lobby, Hall, Office, Storage` is a path of three edges. `Lobby, Hall, Lobby` is a walk, but not a path. `Lobby, Storage` is invalid because it requests a missing direct edge. The sequence's number of edges is one less than its number of listed vertices; its weighted cost is the sum of those edges' weights.

A one-vertex path has zero edges and zero cost if that vertex exists. An empty sequence will be defined as invalid in the route checker later in this chapter. An unknown label is also invalid. These conventions prevent an absent room from being mistaken for a valid isolated room.

**Predict:** add the edge Lab–Office to the diagram. Which degrees change? A cycle Hall, Lab, Office, Hall now exists. Do all vertices become connected? Remote is still isolated. An edge update can change local degree and larger connectivity properties at the same time.

## Simple-Graph Contracts

```{index} simple graph; multigraph; self-loop; graph contract
```

A **simple graph** has no self-loops and at most one edge between a given pair of vertices. A **self-loop** joins a vertex to itself. A **multigraph** may store several distinct edges between the same endpoints, such as separate transport services. Choose the model that matches the domain before selecting storage.

For an undirected simple graph, `(Hall, Lab)` and `(Lab, Hall)` refer to the same logical edge. A raw set of ordered tuples treats them as different tuples unless you normalize their order. Adjacency storage can instead represent the one edge as two reciprocal entries and count it once.

The complete route graph in the next section adopts an explicit contract: vertex labels use ordinal, case-sensitive equality; vertices are added explicitly; both endpoints must exist; self-loops and negative minutes are rejected; repeated edge assignment replaces its weight; removing a vertex removes every incident edge. An isolate stays present until explicitly removed.

Before implementing, identify what the graph omits. A room-level model might ignore doors, stairs, access permissions, or opening hours. Its routes can support decisions only within those assumptions.

## Exercise

```{index} degree sum; exercise
```

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Exercise: compute degrees and check the undirected degree sum.
string[] rooms = { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" };
(string A, string B)[] edges = { ("Lobby", "Hall"), ("Hall", "Lab"), ("Hall", "Office"), ("Office", "Storage") };
var degrees = rooms.ToDictionary(r => r, _ => 0);
// Your code starts here.
/* TODO: count one incidence at each endpoint of every edge. */
// Your code ends here.
Console.WriteLine($"Hall={degrees["Hall"]}, Remote={degrees["Remote"]}");
Console.WriteLine($"degree sum={degrees.Values.Sum()}, twice edges={2 * edges.Length}");

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// Solution
string[] rooms = { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" };
(string A, string B)[] edges = { ("Lobby", "Hall"), ("Hall", "Lab"), ("Hall", "Office"), ("Office", "Storage") };
var degrees = rooms.ToDictionary(r => r, _ => 0);
foreach (var (a, b) in edges) { degrees[a]++; degrees[b]++; }
Console.WriteLine($"Hall={degrees["Hall"]}, Remote={degrees["Remote"]}");
Console.WriteLine($"degree sum={degrees.Values.Sum()}, twice edges={2 * edges.Length}");

Hall=3, Remote=0
degree sum=8, twice edges=8


```{rubric} Footnotes
```

[^1]: Graph-theory texts differ in whether the word path permits repeated vertices. This chapter uses path for a walk with no repeated vertices.
[^2]: With self-loops permitted, an undirected loop contributes two to degree. The simple-graph examples reject loops, so each adjacency entry contributes one.